# 📊 Erasure Coding In Practice

This lab explores the storage efficiency and resilience of Erasure Coding. You'll compare data overhead of different EC configurations versus replication, and inspect the RustFS server's drive health.

In [ ]:
import boto3
import subprocess
import json
import math

# Connect to RustFS S3
s3 = boto3.client(
    service_name='s3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1',
    use_ssl=False,
)

# Create bucket for this lab
s3.create_bucket(Bucket='lab6-ec')
print("Connected to RustFS and created bucket 'lab6-ec'.")

In [ ]:
# Check cluster health before uploading — all 4 drives should be online
result = subprocess.run(
    ['curl', '-sf', 'http://localhost:9000/health'],
    capture_output=True, text=True
)
health_before = json.loads(result.stdout)
print("Health before upload:")
print(json.dumps(health_before, indent=2))

In [ ]:
# Upload objects of varying sizes
sizes = {
    'object_1KB.txt': 1024,
    'object_10KB.txt': 10 * 1024,
    'object_100KB.txt': 100 * 1024,
    'object_1MB.txt': 1024 * 1024,
    'object_10MB.txt': 10 * 1024 * 1024,
}

for key, size in sizes.items():
    content = b'a' * size
    s3.put_object(Bucket='lab6-ec', Key=key, Body=content)
    print(f"Uploaded {key} ({size} bytes)")

In [ ]:
# Check storage metadata for each object
for key in sizes.keys():
    response = s3.head_object(Bucket='lab6-ec', Key=key)
    size = response['ContentLength']
    etag = response['ETag']
    print(f"{key}: size={size} bytes, ETag={etag}")

In [ ]:
# Check health after upload — all drives still online, data striped across them
result = subprocess.run(
    ['curl', '-sf', 'http://localhost:9000/health'],
    capture_output=True, text=True
)
health_after = json.loads(result.stdout)
print("Health after upload:")
print(json.dumps(health_after, indent=2))

drives_online = sum(1 for d in health_after.get('drives', health_after.get('disks', [])) if d.get('status') == 'online')
print(f"\nDrives online: {drives_online} / 4")

In [ ]:
# Calculate and display storage efficiency for various EC configs
configs = [
    ("3x Replication", 1, 3, "33.3%"),
    ("EC:2 (RS 4+2)", 4, 6, f"{4/6:.1%}"),
    ("EC:4 (RS 4+4)", 4, 8, "50.0%"),
    ("EC:6 (RS 6+2)", 6, 8, f"{6/8:.1%}"),
    ("EC:10+4 (RS 10+4)", 10, 14, f"{10/14:.1%}"),
]

print(f"{'Scheme':<20} {'Data':>5} {'Total':>6} {'Efficiency':>11}")
print("-" * 44)
for name, data, total, eff in configs:
    print(f"{name:<20} {data:>5} {total:>6} {eff:>11}")

In [ ]:
# Check the RustFS server container status via Docker
result = subprocess.run(
    ['docker', 'compose', 'ps', '--format', '{{.Name}} {{.Status}}'],
    capture_output=True, text=True
)
print("Container status:")
print(result.stdout)

### 🧪 Simulating Drive Failure

In production, you could simulate drive loss by removing a Docker volume: `docker volume rm <volume>`. The EC:2 scheme (RS 4+2) stripes data + parity across all 4 drives, so losing any 2 still preserves full data access.

Since we cannot unmount drives from the running container in this lab, the key takeaway is:
- The `/health` endpoint reports all drives as `online` when everything is healthy
- If drives were to fail, the RustFS server would continue serving objects using the surviving shards + parity
- Storage efficiency = data_shards / total_shards (4/6 ≈ 67% for EC:2)

In [ ]:
# Concluding markdown table showing EC trade-offs
from IPython.display import display, Markdown

table = """
| EC Scheme | Min Drives | Max Failures | Storage Efficiency | Use Case |
|-----------|-----------|--------------|-------------------|----------|
| 3x Replication | 3 | 2 | 33% | Small clusters, simplicity |
| RS(4,2) | 6 | 2 | 67% | Balanced efficiency & tolerance |
| RS(4,4) | 8 | 4 | 50% | High failure tolerance |
| RS(6,2) | 8 | 2 | 75% | High efficiency, warm storage |
| RS(10,4) | 14 | 4 | 71% | Cold archive, max efficiency |
"""
display(Markdown(table))

In [ ]:
# Cleanup: delete all objects and bucket
response = s3.list_objects_v2(Bucket='lab6-ec')
for obj in response.get('Contents', []):
    s3.delete_object(Bucket='lab6-ec', Key=obj['Key'])
    print(f"Deleted {obj['Key']}")

s3.delete_bucket(Bucket='lab6-ec')
print("Deleted bucket lab6-ec")
print("Cleanup done!")